In [ ]:
# Phase 3: Biographical Profiling
# Notebook: 01_bio_profiling.ipynb
# Owner: Anika
#
# This notebook implements the biographical profiling pipeline:
#   Step 3.1 Define the patient profile schema
#   Step 3.2 Build the query generation module (src/profiling.py)
#   Step 3.3 Build the full profiling-to-retrieval pipeline

import sys
import os
from dotenv import load_dotenv

# Add project root to path so we can import from src/
sys.path.insert(0, os.path.abspath(".."))
load_dotenv("../.env")

## Step 3.1 — Define the patient profile schema

A patient profile is a dictionary with these fields:
- `name`: Patient's full name
- `birth_year`: Year of birth (used to compute the reminiscence bump: ages 15–25)
- `hometown`: City and state (used to target regional music)
- `cultural_background`: Ethnic/cultural identity (used to target genre/artist community)
- `life_events`: List of `{year, event}` dicts mapped to music of those moments

In [ ]:
# Sample patient profile — used for testing throughout this notebook
sample_profile = {
    "name": "James Wilson",
    "birth_year": 1948,
    "hometown": "Detroit, Michigan",
    "cultural_background": "African American",
    "life_events": [
        {"year": 1966, "event": "Graduated from Cass Tech High School"},
        {"year": 1968, "event": "Drafted into the Vietnam War"},
        {"year": 1971, "event": "Married Dorothy in Detroit"},
        {"year": 1975, "event": "First child born"},
        {"year": 1980, "event": "Moved to Atlanta for work"},
    ]
}

# Derived fields (computed automatically in profiling functions)
bump_start = sample_profile["birth_year"] + 15
bump_end   = sample_profile["birth_year"] + 25
print(f"Patient: {sample_profile['name']}")
print(f"Hometown: {sample_profile['hometown']}")
print(f"Cultural background: {sample_profile['cultural_background']}")
print(f"Reminiscence bump: {bump_start}–{bump_end} (ages 15–25)")
print(f"Life events:")
for e in sample_profile["life_events"]:
    print(f"  {e['year']}: {e['event']}")

## Step 3.2 — Query generation module

`generate_queries()` takes a patient profile and uses GPT-4o to produce 5–8 targeted retrieval queries. Each query is crafted to surface songs that are:
- From the patient's reminiscence bump years
- Relevant to their cultural background and geographic region
- Tied to specific life events

The function lives in [src/profiling.py](../src/profiling.py).

In [ ]:
from src.profiling import generate_queries

# Test: generate retrieval queries for the sample profile
queries = generate_queries(sample_profile)

print(f"Generated {len(queries)} queries for {sample_profile['name']}:\n")
for i, q in enumerate(queries, 1):
    print(f"  {i}. {q}")

## Step 3.3 — Full profiling-to-retrieval pipeline

`profile_to_context()` chains everything together:
1. Calls `generate_queries()` to produce targeted queries
2. Runs each query through the FAISS retrieval system
3. Combines and deduplicates results, keeping the top-scored unique songs

**Prerequisites:** The FAISS index (`data/index/songs.index`) and knowledge base (`data/processed/knowledge_base.csv`).

In [ ]:
from src.retrieval import load_retrieval_system
from src.profiling import profile_to_context

INDEX_PATH = "../data/index/songs.index"
KB_PATH    = "../data/processed/knowledge_base.csv"

# Load FAISS index + knowledge base (from Phase 2)
index, df = load_retrieval_system(INDEX_PATH, KB_PATH)
print(f"Loaded index with {index.ntotal} vectors and knowledge base with {len(df)} songs.")

In [ ]:
# Run the full pipeline: profile → queries → retrieved songs
retrieved, queries_used = profile_to_context(sample_profile, index, df, k_per_query=10, total_k=20)

print(f"\nRetrieved {len(retrieved)} unique songs after deduplication:\n")
print(retrieved[["song", "artist", "year", "similarity_score", "source_query"]].to_string(index=False))